In [ ]:
# Final program for Databases and SQL specilization

In [2]:
import prettytable
import pandas as pd
import sqlite3
prettytable.DEFAULT = 'DEFAULT'
pd.set_option('display.max_columns', None)   # show all columns

%load_ext sql

con = sqlite3.connect('FinalDB.db')
census=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01')
census.to_sql('CENSUS_DATA',con)

schools = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01')
crime = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01')
schools.to_sql('CHICAGO_PUBLIC_SCHOOLS',con)
crime.to_sql('CHICAGO_CRIME_DATA',con)

%sql sqlite:///FinalDB.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


ValueError: Table 'CENSUS_DATA' already exists.

In [ ]:
##### Find the total number of crimes recorded in the CRIME table.
%sql select count(*) from CHICAGO_CRIME_DATA;
#533

In [5]:
##### List community area names and numbers with per capita income less than 11000.
%sql select COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER from CENSUS_DATA where PER_CAPITA_INCOME < 11000;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER
West Garfield Park,26.0
South Lawndale,30.0
Fuller Park,37.0
Riverdale,54.0


In [48]:
%%sql
--P3,  List all case numbers for crimes involving minors?(children are not considered minors for the purposes of crime analysis) 
SELECT CASE_NUMBER
FROM CHICAGO_CRIME_DATA
WHERE DESCRIPTION LIKE '%minor%';

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER
HL266884
HK238408


In [27]:
%%sql
--P4,List all kidnapping crimes involving a child?---

SELECT *
FROM CHICAGO_CRIME_DATA
WHERE DESCRIPTION LIKE '%minor%' AND PRIMARY_TYPE = 'KIDNAPPING';

 * sqlite:///FinalDB.db
Done.


index,ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
520,5276766,HN144152,2007-01-26,050XX W VAN BUREN ST,1792,KIDNAPPING,CHILD ABDUCTION/STRANGER,STREET,0,0,1533,15,29.0,25.0,20,1143050.0,1897546.0,2007,41.87490841,-87.75024931,"(41.874908413, -87.750249307)"


In [ ]:
%%sql
--Problem5 List the kind of crimes that were recorded at schools. (No repetitions)

SELECT DISTINCT PRIMARY_TYPE
FROM CHICAGO_CRIME_DATA
WHERE COMMUNITY_AREA_NUMBER IN (
    SELECT COMMUNITY_AREA_NUMBER
    FROM CHICAGO_PUBLIC_SCHOOLS
);

In [49]:
%%sql
--P6, List the type of schools along with the average safety score for each type.

SELECT "Elementary, Middle, or High School",
       AVG(SAFETY_SCORE) AS avg_safety_score
FROM CHICAGO_PUBLIC_SCHOOLS
GROUP BY "Elementary, Middle, or High School";

 * sqlite:///FinalDB.db
Done.


"Elementary, Middle, or High School",avg_safety_score
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


In [ ]:
%%sql
--P7, List 5 community areas with highest % of households below poverty line
SELECT *
FROM CENSUS_DATA
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 5;

In [50]:
%%sql
--P8, Which community area is most crime prone? Display the coumminty area number only
SELECT COMMUNITY_AREA_NUMBER
FROM CHICAGO_CRIME_DATA
GROUP BY COMMUNITY_AREA_NUMBER
ORDER BY COUNT(*) DESC
LIMIT 1;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER
25.0


In [ ]:
%%sql
--P9,Use a sub-query to find the name of the community area with highest hardship index
SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE HARDSHIP_INDEX = (
    SELECT MAX(HARDSHIP_INDEX)
    FROM CENSUS_DATA
);

In [51]:
%%sql
--P10, Use a sub-query to determine the Community Area Name with most number of crimes?
SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER
    FROM CHICAGO_CRIME_DATA
    GROUP BY COMMUNITY_AREA_NUMBER
    ORDER BY COUNT(*) DESC
    LIMIT 1
);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Austin


In [47]:
# crime.to_sql('CHICAGO_CRIME_DATA',con)
#crime['DESCRIPTION'].unique()
# crime['PRIMARY_TYPE'].unique()
# crime.columns
# KIDNAPPING
# crime[crime['DESCRIPTION'].str.contains("minor", case=False, na=False)]['DESCRIPTION'].unique()

# schools.head()
# schools.columns
# schools COMMUNITY_AREA_NUMBER
# 'Elementary, Middle, or High School'
# schools.to_sql('CHICAGO_PUBLIC_SCHOOLS',con)

#COMMUNITY_AREA_NUMBER
# census.head() 
# census.columns
# census.to_sql('CENSUS_DATA',con)

array(['SELL/GIVE/DEL LIQUOR TO MINOR', 'ILLEGAL CONSUMPTION BY MINOR'],
      dtype=object)